In [1]:
import os
import sys


# 获取项目根目录（Lab 的父目录）
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
sys.path.insert(0, project_root)
lib_path = os.path.join(project_root, 'Lib')
sys.path.insert(0, lib_path)

print(f"项目根目录: {project_root}")
print(f"Lib 目录存在: {os.path.exists(os.path.join(project_root, 'Lib'))}")

项目根目录: c:\Users\JoeyC\Desktop\Find_Job_Pipe_Line_V2
Lib 目录存在: True


In [2]:
from Lib.json_yaml_IO import *
from Lib.Html_Analist import HtmlAnalist
from Lib.Batch_Run import BatchRun

In [3]:
from minio import Minio
from minio.error import S3Error

client = Minio(
    endpoint="localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

In [4]:
import threading
import time
import logging
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, TimeoutError as FutureTimeoutError, as_completed
from typing import Callable, List, Any, Optional, Tuple, Dict, Iterable, Union
from dataclasses import dataclass
from enum import Enum

In [5]:
def read_from_s3(bucket_name, object_name):
    response = client.get_object(bucket_name, object_name)
    content = response.read()
    response.close()
    return content

In [6]:
bucket_name = "jobdatabucket"
html_dir = "raw/Linkedin_html/dt=2025-12-19"

contents = []
objects = client.list_objects(bucket_name, prefix=html_dir, recursive=True)

start_time = time.time()
# 处理完成的任务
completed_count = 0
with ThreadPoolExecutor(max_workers=15) as executor:
    futures = []
    # 提交所有任务
    for obj in objects:
        future = executor.submit(read_from_s3, bucket_name, obj.object_name)
        futures.append(future)
        
    for future in as_completed(futures):
        try:
            content = future.result()
        except Exception as e:
            print(f"Error processing: {e}")
            
        contents.append(content)
        completed_count += 1

        print(f"\rProgress: {completed_count}", end="", flush=True)
        print(f"Time taken: {time.time() - start_time:.2f} seconds")

Progress: 1Time taken: 0.20 seconds
Progress: 2Time taken: 0.21 seconds
Progress: 3Time taken: 0.22 seconds
Progress: 4Time taken: 0.25 seconds
Progress: 5Time taken: 0.26 seconds
Progress: 6Time taken: 0.28 seconds
Progress: 7Time taken: 0.30 seconds
Progress: 8Time taken: 0.31 seconds
Progress: 9Time taken: 0.33 seconds
Progress: 10Time taken: 0.34 seconds
Progress: 11Time taken: 0.36 seconds
Progress: 12Time taken: 0.37 seconds
Progress: 13Time taken: 0.39 seconds
Progress: 14Time taken: 0.40 seconds
Progress: 15Time taken: 0.41 seconds
Progress: 16Time taken: 0.43 seconds
Progress: 17Time taken: 0.43 seconds
Progress: 18Time taken: 0.45 seconds
Progress: 19Time taken: 0.46 seconds
Progress: 20Time taken: 0.47 seconds
Progress: 21Time taken: 0.48 seconds
Progress: 22Time taken: 0.49 seconds
Progress: 23Time taken: 0.50 seconds
Progress: 24Time taken: 0.52 seconds
Progress: 25Time taken: 0.54 seconds
Progress: 26Time taken: 0.55 seconds
Progress: 27Time taken: 0.56 seconds
Progress: 

In [7]:
def extract_location_and_time(text):
    import re
    try:
        # 北上广深直接返回
        if '北京' in text:
            location = '北京'
        elif '上海' in text:
            location = '上海'
        elif '广州' in text:
            location = '广州'
        elif '深圳' in text:
            location = '深圳'
        else:
            # 其他省份
            provinces = ['河北', '山西', '辽宁', '吉林', '黑龙江', '江苏', '浙江', 
                        '安徽', '福建', '江西', '山东', '河南', '湖北', '湖南', 
                        '广东', '海南', '四川', '贵州', '云南', '陕西', '甘肃', 
                        '青海', '内蒙古', '广西', '西藏', '宁夏', '新疆']
        
            for province in provinces:
                if province in text:
                    location = province
            if not location:
                location = None
        
        time_pattern = re.compile(r'(?<!\d)(?:(?:\d{1,4}\s*(?:分钟|小时|天|周|个月|年))|(?:今天|昨天|前天)\s*\d{1,2}:\d{2})|(?:\d{1,2}\s*秒前)')
        time_match = time_pattern.search(text)
        if time_match:
            time = time_match.group(0)
        else:
            time = None
    except Exception as e:
        print(f"Error extracting location and time: {e}")
        location = None
        time = None

    return location, time

In [8]:
cpu_count = os.cpu_count()
print(f"CPU 核心数: {cpu_count}")

CPU 核心数: 16


In [9]:
def process_file(content: bytes) -> Any:
    html_processor = HtmlAnalist(content)

    company_url_and_name = html_processor.get_company_url_and_name()
    company_url = company_url_and_name['company_url']
    company_name = company_url_and_name['company_name']

    info = html_processor.info
    location, time = extract_location_and_time(info)

    result = {
        'job_name': html_processor.head,
        'job_url': html_processor.head_url,
        'company_name': company_name,
        'company_url': company_url,
        'location': location,
        'time': time,
        'job_details_content': html_processor.get_job_details_content(),
        'company_box_content': html_processor.get_company_box_content()
    }

    print(f'Processing: {result["company_name"]} - {result["job_name"]}')
    return result

In [10]:
processed_contents = []
for raw_content in contents:
    processed_content = process_file(raw_content)
    processed_contents.append(processed_content)

Processing: 深圳天源迪科信息技术股份有限公司 - 数据库开发 （源码）
Processing: 天津瑞湾国际商务中心有限公司 - 算法工程师
Processing: （株）同和 - 数据治理工程师
Processing: 新希望集团有限公司 - 数据分析岗
Processing: T.P.R. LIMITED - 生活服务-数据产品专家
Processing: JSmart  - CRM数据分析副经理
Processing: 搜房网 - 数据分析师
Processing: 结构性融资与资产证券化 - 金融数据分析实习生
Processing: Shanghai Wondertek Software. Co., Ltd.(网达软件) - 数据治理工程师
Processing: 中国平安 - 26619J-初级数据分析岗
Processing: 贝壳找房ke.com - 数据分析师
Processing: 国双 - 数据产品(J13287)
Processing: ShowMe - 数据分析师
Processing: 凯悦酒店 - Gerente assistente/líder de equipe - sem detalhes pessoais
Processing: JSmart  - CRM数据分析副经理
Processing: 麦可思公司 - 数据分析师
Processing: 广州广颖创网络科技有限公司 - 数据分析专员
Processing: Bocomm Leasing - 数据分析工程师
Processing: 上海沧溟贸易有限公司 - 大数据分析工程师
Processing: 联合服务（香港）有限公司 - 数据分析师
Processing: 浙江深大智能科技有限公司 - 数据运营
Processing: BIGO - Bigo live-数据分析师
Processing: Digital Heaven Information&Technology Co. Ltd. - 数据分析师
Processing: 哈尔滨云计算中心有限公司 - 大数据分析师
Processing: 马上消费金融股份有限公司 - 大数据/数据仓库工程师
Processing: 证金人力资源 - 数据架构师
Processing: 知乎 - 数据分析
Processing

In [11]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from io import BytesIO
from datetime import datetime

In [12]:
def save_json_to_s3(
    processed_content, bucket_name, object_path):
    data = json.dumps(processed_content, indent=4).encode('utf-8')
    data_stream = BytesIO(data)
    content_name = f'{processed_content['company_name']}_{processed_content['job_name']}.json'
    try:
        client.put_object(
            bucket_name = bucket_name, 
            object_name = f'{object_path}/{content_name}', 
            data = data_stream,
            length = len(data),
            content_type = 'application/json'
        )
    except Exception as e:
        print(f"Error saving json to s3: {e}")

    return content_name

In [13]:
bucket_name = "jobdatabucket"
object_path = "raw/raw_json/dt=2025-12-19"
start_time = time.time()
with ThreadPoolExecutor(max_workers=15) as executor:
    futures = []
    # 提交所有任务
    for processed_content in processed_contents:
        content_name = f'{processed_content['company_name']}_{processed_content['job_name']}.json'

        future = executor.submit(
            save_json_to_s3, 
            processed_content, 
            bucket_name, 
            object_path
        )
        futures.append(future)  

    for future in as_completed(futures):
        try:
            content_name = future.result()
            if content_name:
                print(f"Saved: {content_name}")
        except Exception as e:
            print(f"Error saving json to s3: {e}")

        print(f"Time taken: {time.time() - start_time:.2f} seconds")

Saved: （株）同和_数据治理工程师.json
Time taken: 0.08 seconds
Saved: 搜房网_数据分析师.json
Time taken: 0.08 seconds
Saved: 新希望集团有限公司_数据分析岗.json
Time taken: 0.08 seconds
Saved: 结构性融资与资产证券化_金融数据分析实习生.json
Time taken: 0.08 seconds
Saved: T.P.R. LIMITED_生活服务-数据产品专家.json
Time taken: 0.08 seconds
Saved: JSmart _CRM数据分析副经理.json
Time taken: 0.09 seconds
Saved: 中国平安_26619J-初级数据分析岗.json
Time taken: 0.09 seconds
Saved: 深圳天源迪科信息技术股份有限公司_数据库开发 （源码）.json
Time taken: 0.09 seconds
Saved: 贝壳找房ke.com_数据分析师.json
Time taken: 0.09 seconds
Saved: 天津瑞湾国际商务中心有限公司_算法工程师.json
Time taken: 0.09 seconds
Saved: ShowMe_数据分析师.json
Time taken: 0.09 seconds
Saved: Shanghai Wondertek Software. Co., Ltd.(网达软件)_数据治理工程师.json
Time taken: 0.09 seconds
Saved: 国双_数据产品(J13287).json
Time taken: 0.09 seconds
Saved: 凯悦酒店_Gerente assistente/líder de equipe - sem detalhes pessoais.json
Time taken: 0.10 seconds
Saved: JSmart _CRM数据分析副经理.json
Time taken: 0.11 seconds
Saved: 麦可思公司_数据分析师.json
Time taken: 0.12 seconds
Saved: 广州广颖创网络科技有限公司_数据分析专员.json
Time